# M13 Portfolio Data Collection Pipeline

**Purpose**: Demonstrates the data engineering approach behind the M13 Deal Brief tool.  
This notebook shows how portfolio company data is collected, cleaned, and structured for the briefing generation system.

**Skills demonstrated**: Web scraping, API integration, data normalization, pandas, visualization

---

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

# For web scraping (demo purposes)
# import requests
# from bs4 import BeautifulSoup

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

## 1. Load Curated Portfolio Data

In production, this data would be pulled from CRM (Affinity/DealCloud), Crunchbase API, and internal deal memos.  
For this demo, we use our curated JSON dataset.

In [ ]:
# Simulating the curated data (matches our Next.js data layer)
companies_raw = [
    {
        "name": "Lyft", "sector": "Mobility / Transportation", "stage": "Public",
        "founded": 2012, "hq": "San Francisco, CA",
        "total_raised": 4500, "employees_est": 4000, "revenue_est": 4400,
        "funding_rounds": 6, "exit_type": "IPO", "exit_value": 2300
    },
    {
        "name": "Pinterest", "sector": "Commerce / Social Discovery", "stage": "Public",
        "founded": 2010, "hq": "San Francisco, CA",
        "total_raised": 1500, "employees_est": 5500, "revenue_est": 3600,
        "funding_rounds": 5, "exit_type": "IPO", "exit_value": 1400
    },
    {
        "name": "Ring", "sector": "IoT / Smart Home Security", "stage": "Acquired",
        "founded": 2013, "hq": "Santa Monica, CA",
        "total_raised": 209, "employees_est": 2000, "revenue_est": 1800,
        "funding_rounds": 5, "exit_type": "Acquisition", "exit_value": 1200
    },
    {
        "name": "Rho", "sector": "Fintech / B2B Finance", "stage": "Series C",
        "founded": 2018, "hq": "New York, NY",
        "total_raised": 224, "employees_est": 350, "revenue_est": 50,
        "funding_rounds": 4, "exit_type": None, "exit_value": None
    },
    {
        "name": "Matterport", "sector": "PropTech / Spatial Computing", "stage": "Acquired",
        "founded": 2011, "hq": "Sunnyvale, CA",
        "total_raised": 690, "employees_est": 600, "revenue_est": 160,
        "funding_rounds": 5, "exit_type": "Acquisition", "exit_value": 1600
    },
    {
        "name": "Prepared", "sector": "GovTech / Public Safety", "stage": "Series B",
        "founded": 2018, "hq": "New York, NY",
        "total_raised": 47, "employees_est": 100, "revenue_est": 20,
        "funding_rounds": 3, "exit_type": None, "exit_value": None
    }
]

df = pd.DataFrame(companies_raw)
print(f"Portfolio: {len(df)} companies loaded")
df

## 2. Data Cleaning & Normalization

Real-world VC data is messy — inconsistent formats, missing fields, conflicting sources.  
Here we standardize everything into a consistent schema.

In [ ]:
# Normalize revenue to millions
df['revenue_est_M'] = df['revenue_est']
df['total_raised_M'] = df['total_raised']

# Calculate capital efficiency (revenue / total raised)
df['capital_efficiency'] = (df['revenue_est'] / df['total_raised']).round(2)

# Company age
df['age_years'] = 2025 - df['founded']

# Revenue per employee (rough efficiency metric)
df['rev_per_employee_K'] = ((df['revenue_est'] * 1000) / df['employees_est']).round(0)

print("\n=== Capital Efficiency (Revenue / Total Raised) ===")
for _, row in df.sort_values('capital_efficiency', ascending=False).iterrows():
    print(f"  {row['name']:12s}  {row['capital_efficiency']:.2f}x")

print(f"\n=== Revenue per Employee ===")
for _, row in df.sort_values('rev_per_employee_K', ascending=False).iterrows():
    print(f"  {row['name']:12s}  ${row['rev_per_employee_K']:.0f}K")

## 3. Portfolio Analysis — Sector Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sector distribution
sector_counts = df['sector'].value_counts()
colors = ['#3b82f6', '#ec4899', '#10b981', '#f59e0b', '#8b5cf6', '#ef4444']
axes[0].barh(sector_counts.index, sector_counts.values, color=colors[:len(sector_counts)])
axes[0].set_title('Portfolio by Sector', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Companies')

# Stage distribution
stage_counts = df['stage'].value_counts()
axes[1].pie(stage_counts.values, labels=stage_counts.index, autopct='%1.0f%%',
            colors=['#6366f1', '#22c55e', '#f97316'], startangle=90)
axes[1].set_title('Portfolio by Stage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('portfolio_analysis.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()

## 4. Funding Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Sort by total raised
df_sorted = df.sort_values('total_raised_M', ascending=True)

bars = ax.barh(df_sorted['name'], df_sorted['total_raised_M'], color='#6366f1', alpha=0.8)

# Add value labels
for bar, val in zip(bars, df_sorted['total_raised_M']):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2, 
            f'${val:,.0f}M', va='center', fontsize=10, color='white')

ax.set_title('Total Capital Raised by Portfolio Company', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Raised ($M)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('funding_analysis.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()

## 5. Exit Analysis

In [ ]:
exited = df[df['exit_value'].notna()].copy()
exited['return_multiple'] = (exited['exit_value'] / exited['total_raised']).round(2)

print("=== Exit Performance ===")
print(f"Companies with exits: {len(exited)}/{len(df)}")
print(f"Total exit value: ${exited['exit_value'].sum():,.0f}M")
print(f"Average return multiple: {exited['return_multiple'].mean():.1f}x\n")

for _, row in exited.iterrows():
    print(f"  {row['name']:12s}  {row['exit_type']:12s}  ${row['exit_value']:,.0f}M  ({row['return_multiple']:.1f}x on ${row['total_raised']:,.0f}M raised)")

# Scatter plot: raised vs exit value
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(exited['total_raised_M'], exited['exit_value'], 
                     s=exited['employees_est']/5, alpha=0.7, 
                     c=['#6366f1', '#22c55e', '#f97316'][:len(exited)],
                     edgecolors='white', linewidth=0.5)

for _, row in exited.iterrows():
    ax.annotate(row['name'], (row['total_raised_M'], row['exit_value']),
                textcoords="offset points", xytext=(10, 5), fontsize=11, color='white')

ax.set_xlabel('Total Raised ($M)', fontsize=12)
ax.set_ylabel('Exit Value ($M)', fontsize=12)
ax.set_title('Capital Raised vs Exit Value (bubble = team size)', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('exit_analysis.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()

## 6. Web Scraping Demo — Collecting News Data

In the live app, we fetch recent news via Serper API. Here's how the raw scraping approach would work for a custom pipeline.

In [ ]:
# --- Web scraping example (commented out for portability) ---
#
# def scrape_company_news(company_name: str, num_results: int = 5) -> list[dict]:
#     """Scrape recent news articles for a company."""
#     headers = {'User-Agent': 'Mozilla/5.0 (compatible; M13Bot/1.0)'}
#     url = f"https://news.google.com/rss/search?q={company_name}+company&hl=en-US"
#     
#     response = requests.get(url, headers=headers, timeout=10)
#     soup = BeautifulSoup(response.content, 'xml')
#     
#     articles = []
#     for item in soup.find_all('item')[:num_results]:
#         articles.append({
#             'title': item.title.text,
#             'link': item.link.text,
#             'published': item.pubDate.text,
#             'source': item.source.text if item.source else 'Unknown'
#         })
#     return articles
#
# # Example usage:
# news = scrape_company_news("Rho fintech")
# for article in news:
#     print(f"[{article['source']}] {article['title']}")

# Simulated output for demo:
simulated_news = [
    {"source": "TechCrunch", "title": "Rho raises $130M Series C to expand corporate finance platform", "date": "2024-09-15"},
    {"source": "Forbes", "title": "Prepared Named to AI 50 List for 911 Technology Innovation", "date": "2024-06-20"},
    {"source": "Bloomberg", "title": "CoStar Completes $1.6B Acquisition of Matterport", "date": "2024-04-12"},
    {"source": "CNBC", "title": "Lyft Reports First GAAP Profit as Rideshare Market Stabilizes", "date": "2024-12-01"},
    {"source": "WSJ", "title": "Pinterest's Ad Revenue Surges on AI-Powered Shopping Features", "date": "2024-11-08"},
]

news_df = pd.DataFrame(simulated_news)
print("=== Recent Portfolio News (Simulated) ===")
for _, article in news_df.iterrows():
    print(f"  [{article['source']:12s}] {article['title']}")

## 7. Data Export — JSON for Web App

The final step in the pipeline: export cleaned, structured data to JSON files that the Next.js app consumes.

In [ ]:
# Export portfolio summary
portfolio_summary = {
    "generated_at": datetime.now().isoformat(),
    "total_companies": len(df),
    "sectors": df['sector'].nunique(),
    "total_capital_deployed": f"${df['total_raised_M'].sum():,.0f}M",
    "exits": len(exited),
    "total_exit_value": f"${exited['exit_value'].sum():,.0f}M",
    "companies": df[['name', 'sector', 'stage', 'founded', 'total_raised_M', 'capital_efficiency']].to_dict('records')
}

# Save to JSON
with open('portfolio_summary.json', 'w') as f:
    json.dump(portfolio_summary, f, indent=2)

print("Portfolio summary exported to portfolio_summary.json")
print(json.dumps(portfolio_summary, indent=2))

---

## What This Pipeline Would Look Like With M13's Internal Data

With access to M13's actual systems, this pipeline transforms significantly:

1. **CRM Integration (Affinity/DealCloud)** → Pull real-time deal data, contact history, meeting notes
2. **Financial Data APIs (PitchBook, CB Insights)** → Automated valuation tracking, comparable analysis
3. **Portfolio Reporting** → Quarterly metrics from companies (MRR, burn rate, runway)
4. **Calendar Integration** → Auto-trigger briefing generation before scheduled meetings
5. **Embeddings + Vector Search** → Semantic search across all internal notes, memos, and documents

The architecture is designed to be data-source agnostic — swap the JSON files for API calls and the system scales immediately.